# Team Composition and Remaining Batting Depth Analysis

This notebook calculates team composition and remaining batting depth metrics for each section of first innings in T20 matches, and stores them in the `first_innings_sections_clean` table.

## Overview

For each section (2-over intervals) in the first innings, this script calculates:

1. **Team Composition**: The total count of players by classification in the team lineup
   - Specialist Batters (BATTER classification)
   - All-rounders (ALL_ROUNDER classification)
   - Bowlers (BOWLER classification)

2. **Remaining Batting Depth**: The count of players by classification who have not been dismissed up to that section
   - Specialist Batters remaining
   - All-rounders remaining
   - Bowlers remaining

3. **Remaining Batting Strength Score**: A weighted score representing the batting strength of remaining players
   - Calculated as the sum of weights for all remaining players
   - Weights are assigned based on player classification

## Player Classification Weights

The following weights are used to calculate the Remaining Batting Strength Score:

- **BATTER**: 1.0 (Full batting value)
- **ALL_ROUNDER**: 0.8 (High batting value)
- **UTILITY_PLAYER**: 0.65 (Moderate batting value)
- **BOWLER**: 0.3 (Low batting value)
- **INSUFFICIENT_DATA**: 0.45 (Default for unclassified players)

## Data Sources

- **first_innings_sections_clean**: Table containing section-by-section data for first innings
- **player_classifications**: Table containing player classifications (BATTER, ALL_ROUNDER, BOWLER, etc.)
- **JSON match files**: Raw match data in `t20s_json/male/` directory containing team lineups and dismissal information

## Methodology

1. **Match ID Construction**: Match IDs are constructed from date, teams, and venue: `{date}|{team1} vs {team2}|{venue}`

2. **Section Mapping**: 
   - Section 1 = overs 0-1
   - Section 2 = overs 2-3
   - Section N = overs (N*2-2) to (N*2-1)

3. **Dismissal Tracking**: For each section, the script identifies all players dismissed up to the end of that section by examining wicket deliveries in the first innings.

4. **Team Composition Calculation**: Counts all players in the team lineup by their classification.

5. **Remaining Batting Depth Calculation**: 
   - Identifies players not yet dismissed at each section
   - Counts remaining players by classification

6. **Batting Strength Score Calculation**:
   - For each remaining player, retrieves their classification
   - Sums the weights of all remaining players
   - Rounds to 2 decimal places

## Output Columns

The following columns are added to `first_innings_sections_clean`:

- **team_composition** (TEXT): JSON string containing counts of specialist_batters, all_rounders, and bowlers
  - Example: `{"specialist_batters": 3, "all_rounders": 2, "bowlers": 6}`

- **remaining_batting_depth** (TEXT): JSON string containing counts of remaining players by classification
  - Example: `{"specialist_batters": 2, "all_rounders": 1, "bowlers": 6}`

- **remaining_batting_strength_score** (REAL): Numeric score representing batting strength of remaining players
  - Example: `4.9` (sum of weights for all remaining players)

- **final_innings_score** (INTEGER): Total runs scored in the first innings for the team
  - Example: `173` (final total runs in the first innings)

## Example Calculation

If a team has:
- 3 BATTERs (all remaining): 3 × 1.0 = 3.0
- 2 ALL_ROUNDERs (1 dismissed, 1 remaining): 1 × 0.8 = 0.8
- 6 BOWLERs (all remaining): 6 × 0.3 = 1.8

**Remaining Batting Strength Score** = 3.0 + 0.8 + 1.8 = **5.6**


In [ ]:
# Calculate team composition and remaining batting depth
import sqlite3
import pandas as pd
import json
from pathlib import Path
import sys
from collections import defaultdict
from typing import Dict, List, Tuple

# Add project root to path to import forecaster
current_dir = Path().resolve()
if current_dir.name == 'research':
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.insert(0, str(project_root))

# Import MatchData
import importlib
for mod in ['forecaster.models', 'forecaster']:
    if mod in sys.modules:
        del sys.modules[mod]
from forecaster.models import MatchData

# Connect to database
db_path = str(project_root / 't20s_json' / 'venue_metrics.db')
conn = sqlite3.connect(db_path)

# Load player classifications
player_classifications_df = pd.read_sql_query("SELECT player, classification FROM player_classifications", conn)
player_class_map = dict(zip(player_classifications_df['player'], player_classifications_df['classification']))

# Load first_innings_sections_clean
sections_df = pd.read_sql_query("SELECT * FROM first_innings_sections_clean", conn)

print(f"Loaded {len(sections_df)} rows from first_innings_sections_clean")
print(f"Loaded {len(player_class_map)} player classifications")

# Load all JSON match files and create match_id to match mapping
male_dir = project_root / 't20s_json' / 'male'
all_json_files = sorted(male_dir.glob('*.json'))

print(f"\nLoading {len(all_json_files)} JSON match files...")

match_data_map = {}  # match_id -> MatchData

for json_file_path in all_json_files:
    try:
        with open(json_file_path, 'r') as f:
            json_data = json.load(f)
        
        match_data = MatchData.model_validate(json_data)
        
        # Create match_id from date, teams, and venue
        date = match_data.info.dates[0] if match_data.info.dates else 'unknown'
        teams = ' vs '.join(match_data.info.teams) if match_data.info.teams else 'unknown'
        venue = match_data.info.venue or 'unknown'
        match_id = f"{date}|{teams}|{venue}"
        
        match_data_map[match_id] = match_data
    except Exception as e:
        print(f"Error processing {json_file_path.name}: {e}")

print(f"Loaded {len(match_data_map)} matches into memory")

# Helper function to get dismissed players up to a certain section
def get_dismissed_players(match_data: MatchData, team: str, section: int) -> set:
    """Get set of players dismissed up to the end of the given section (2 overs per section)."""
    dismissed = set()
    
    if not match_data.innings:
        return dismissed
    
    first_innings = match_data.innings[0]
    if first_innings.team != team:
        return dismissed
    
    # Section 1 = overs 0-1, Section 2 = overs 2-3, etc.
    # Section N ends at over (N*2 - 1)
    max_over = section * 2 - 1
    
    for over in first_innings.overs:
        if over.over > max_over:
            break
        for delivery in over.deliveries:
            if delivery.wickets:
                for wicket in delivery.wickets:
                    dismissed.add(wicket.player_out)
    
    return dismissed

# Helper function to classify player
def get_player_classification(player_name: str) -> str:
    """Get player classification, defaulting to INSUFFICIENT_DATA if not found."""
    return player_class_map.get(player_name, 'INSUFFICIENT_DATA')

# Helper function to count by classification
def count_by_classification(players: List[str], classification: str) -> int:
    """Count players with the given classification."""
    return sum(1 for p in players if get_player_classification(p) == classification)

# Weights for batting strength calculation
weights = {
    "BATTER": 1.0,
    "ALL_ROUNDER": 0.8,
    "UTILITY_PLAYER": 0.65,
    "BOWLER": 0.3,
    "INSUFFICIENT_DATA": 0.45
}

# Helper function to calculate remaining batting strength score
def calculate_batting_strength_score(remaining_players: List[str]) -> float:
    """Calculate the batting strength score based on remaining players and their classifications."""
    score = 0.0
    for player in remaining_players:
        classification = get_player_classification(player)
        weight = weights.get(classification, 0.45)  # Default to INSUFFICIENT_DATA weight
        score += weight
    return round(score, 2)

# Helper function to get final innings score
def get_final_innings_score(match_data: MatchData, team: str) -> int:
    """Get the final total runs scored in the first innings for the given team."""
    if not match_data.innings:
        return None
    
    first_innings = match_data.innings[0]
    if first_innings.team != team:
        return None
    
    total_runs = 0
    for over in first_innings.overs:
        for delivery in over.deliveries:
            total_runs += delivery.runs.total
    
    return total_runs

# Calculate team composition and remaining batting depth for each row
print("\nCalculating team composition and remaining batting depth...")

team_composition_list = []
remaining_batting_depth_list = []
remaining_batting_strength_score_list = []
final_innings_score_list = []

for idx, row in sections_df.iterrows():
    match_id = row['match_id']
    team = row['team']
    section = row['section']
    
    # Get match data
    match_data = match_data_map.get(match_id)
    
    if match_data is None:
        # If match not found, set to None
        team_composition_list.append(None)
        remaining_batting_depth_list.append(None)
        remaining_batting_strength_score_list.append(None)
        final_innings_score_list.append(None)
        continue
    
    # Get team lineup from JSON
    team_players = match_data.info.players.get(team, [])
    
    if not team_players:
        team_composition_list.append(None)
        remaining_batting_depth_list.append(None)
        remaining_batting_strength_score_list.append(None)
        final_innings_score_list.append(None)
        continue
    
    # Get final innings score (same for all sections of the same match)
    final_score = get_final_innings_score(match_data, team)
    
    # Calculate team composition (all players in the team)
    team_composition = {
        'specialist_batters': count_by_classification(team_players, 'BATTER'),
        'all_rounders': count_by_classification(team_players, 'ALL_ROUNDER'),
        'bowlers': count_by_classification(team_players, 'BOWLER')
    }
    
    # Get dismissed players up to this section
    dismissed = get_dismissed_players(match_data, team, section)
    
    # Calculate remaining batting depth (players not yet dismissed)
    remaining_players = [p for p in team_players if p not in dismissed]
    remaining_batting_depth = {
        'specialist_batters': count_by_classification(remaining_players, 'BATTER'),
        'all_rounders': count_by_classification(remaining_players, 'ALL_ROUNDER'),
        'bowlers': count_by_classification(remaining_players, 'BOWLER')
    }
    
    # Calculate remaining batting strength score
    batting_strength_score = calculate_batting_strength_score(remaining_players)
    
    # Store as JSON strings
    team_composition_list.append(json.dumps(team_composition))
    remaining_batting_depth_list.append(json.dumps(remaining_batting_depth))
    remaining_batting_strength_score_list.append(batting_strength_score)
    final_innings_score_list.append(final_score)

# Add new columns to dataframe
sections_df['team_composition'] = team_composition_list
sections_df['remaining_batting_depth'] = remaining_batting_depth_list
sections_df['remaining_batting_strength_score'] = remaining_batting_strength_score_list
sections_df['final_innings_score'] = final_innings_score_list

# Count non-null values
non_null_composition = sum(1 for x in team_composition_list if x is not None)
non_null_depth = sum(1 for x in remaining_batting_depth_list if x is not None)
non_null_score = sum(1 for x in remaining_batting_strength_score_list if x is not None)
non_null_final_score = sum(1 for x in final_innings_score_list if x is not None)

print(f"\nCalculated team composition for {non_null_composition} rows")
print(f"Calculated remaining batting depth for {non_null_depth} rows")
print(f"Calculated remaining batting strength score for {non_null_score} rows")
print(f"Calculated final innings score for {non_null_final_score} rows")

# Update database table
print("\nUpdating database...")

# First, add columns if they don't exist
try:
    conn.execute("ALTER TABLE first_innings_sections_clean ADD COLUMN team_composition TEXT")
except sqlite3.OperationalError as e:
    if "duplicate column" not in str(e).lower():
        raise
    print("  Column team_composition already exists")

try:
    conn.execute("ALTER TABLE first_innings_sections_clean ADD COLUMN remaining_batting_depth TEXT")
except sqlite3.OperationalError as e:
    if "duplicate column" not in str(e).lower():
        raise
    print("  Column remaining_batting_depth already exists")

try:
    conn.execute("ALTER TABLE first_innings_sections_clean ADD COLUMN remaining_batting_strength_score REAL")
except sqlite3.OperationalError as e:
    if "duplicate column" not in str(e).lower():
        raise
    print("  Column remaining_batting_strength_score already exists")

try:
    conn.execute("ALTER TABLE first_innings_sections_clean ADD COLUMN final_innings_score INTEGER")
except sqlite3.OperationalError as e:
    if "duplicate column" not in str(e).lower():
        raise
    print("  Column final_innings_score already exists")

# Update rows
for idx, row in sections_df.iterrows():
    match_id = row['match_id']
    section = row['section']
    team_composition = row['team_composition']
    remaining_batting_depth = row['remaining_batting_depth']
    remaining_batting_strength_score = row['remaining_batting_strength_score']
    final_innings_score = row['final_innings_score']
    
    conn.execute("""
        UPDATE first_innings_sections_clean 
        SET team_composition = ?, remaining_batting_depth = ?, remaining_batting_strength_score = ?, final_innings_score = ?
        WHERE match_id = ? AND section = ?
    """, (team_composition, remaining_batting_depth, remaining_batting_strength_score, final_innings_score, match_id, section))

conn.commit()

print("✓ Updated first_innings_sections_clean table with team_composition, remaining_batting_depth, remaining_batting_strength_score, and final_innings_score columns")

# Show sample results
print("\nSample results:")
sample_df = sections_df[['match_id', 'team', 'section', 'cumulative_wickets', 'team_composition', 'remaining_batting_depth', 'remaining_batting_strength_score', 'final_innings_score']].head(10)
for idx, row in sample_df.iterrows():
    print(f"\nMatch: {row['match_id'][:50]}...")
    print(f"  Team: {row['team']}, Section: {row['section']}, Wickets: {row['cumulative_wickets']}")
    if row['team_composition']:
        comp = json.loads(row['team_composition'])
        print(f"  Team Composition: {comp}")
    if row['remaining_batting_depth']:
        depth = json.loads(row['remaining_batting_depth'])
        print(f"  Remaining Batting Depth: {depth}")
    if row['remaining_batting_strength_score'] is not None:
        print(f"  Remaining Batting Strength Score: {row['remaining_batting_strength_score']}")
    if row['final_innings_score'] is not None:
        print(f"  Final Innings Score: {row['final_innings_score']}")

conn.close()


Loaded 28954 rows from first_innings_sections_clean
Loaded 4223 player classifications

Loading 3112 JSON match files...
Loaded 3065 matches into memory

Calculating team composition and remaining batting depth...

Calculated team composition for 28954 rows
Calculated remaining batting depth for 28954 rows
Calculated remaining batting strength score for 28954 rows
Calculated final innings score for 28944 rows

Updating database...
  Column team_composition already exists
  Column remaining_batting_depth already exists
  Column remaining_batting_strength_score already exists
✓ Updated first_innings_sections_clean table with team_composition, remaining_batting_depth, remaining_batting_strength_score, and final_innings_score columns

Sample results:

Match: 2017-02-19|Australia vs Sri Lanka|Simonds Stadium,...
  Team: Australia, Section: 1, Wickets: 0
  Team Composition: {'specialist_batters': 2, 'all_rounders': 0, 'bowlers': 1}
  Remaining Batting Depth: {'specialist_batters': 2, 'all_ro

In [1]:
import sqlite3
from pathlib import Path

# Connect to database
current_dir = Path().resolve()
project_root = current_dir.parent if current_dir.name == 'research' else current_dir
db_path = str(project_root / 't20s_json' / 'venue_metrics.db')
conn = sqlite3.connect(db_path)

# Add cumulative_run_rate_rounded column (ROUND to 2 decimal places)
try:
    conn.execute("ALTER TABLE first_innings_sections_clean ADD COLUMN cumulative_run_rate_rounded REAL")
except sqlite3.OperationalError as e:
    if "duplicate column" not in str(e).lower():
        raise
    print("  Column cumulative_run_rate_rounded already exists")

conn.execute("UPDATE first_innings_sections_clean SET cumulative_run_rate_rounded = ROUND(cumulative_run_rate, 2)")
conn.commit()

print("✓ Added 'cumulative_run_rate_rounded' column to first_innings_sections_clean")

# Verify
sample = conn.execute(
    "SELECT cumulative_run_rate, cumulative_run_rate_rounded FROM first_innings_sections_clean LIMIT 5"
).fetchall()
print("\nSample (cumulative_run_rate → cumulative_run_rate_rounded):")
for row in sample:
    print(f"  {row[0]} → {row[1]}")

conn.close()


✓ Added 'cumulative_run_rate_rounded' column to first_innings_sections_clean

Sample (cumulative_run_rate → cumulative_run_rate_rounded):
  8.5 → 8.5
  7.5 → 7.5
  10.0 → 10.0
  8.375 → 8.38
  8.0 → 8.0
